# Day 063 — Exercise 5: Stripe Checkout (Mocked)

Stripe Checkout is a Stripe-hosted payment page. Your server creates a **checkout session** and redirects the user to the Stripe URL. After payment, Stripe redirects back to your success URL.

```
POST /checkout {"plan": "pro"}
    → 200 {"session_id": "cs_test_...", "checkout_url": "https://checkout.stripe.com/..."}
    → 302/redirect to checkout_url (in a browser)
```

For testing, the `stripe_client` is injected — replace the real `stripe` module with a duck-typed mock that returns predictable data.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from starlette.testclient import TestClient

# Duck-typed mock: any object with .checkout.Session.create(**kwargs) works
class _MockStripe:
    class checkout:
        class Session:
            @staticmethod
            def create(**kwargs):
                return {"id": "cs_test_abc123",
                        "url": "https://checkout.stripe.com/pay/test"}

PLAN_PRICES = {
    "pro": "price_pro_monthly",
}


## Task

Implement `build_checkout_api(stripe_client=None) -> FastAPI`:

```
POST /checkout  {"plan": str}  → {"session_id", "checkout_url"}  or 400
GET /checkout/cancel           → {"message": "Checkout cancelled..."}
```

- Use `stripe_client.checkout.Session.create(**kwargs)` to get the session
- If `plan` not in `PLAN_PRICES`: raise `HTTPException(400)`
- If `stripe_client` is `None`: use `_MockStripe()`

## Your Implementation

In [ ]:
def build_checkout_api(stripe_client=None) -> FastAPI:
    """FastAPI with Stripe checkout endpoints.

    POST /checkout  {"plan": str}
        → 200 {"session_id": str, "checkout_url": str}
        → 400 if plan not in PLAN_PRICES
        Uses stripe_client.checkout.Session.create(
            payment_method_types=['card'],
            line_items=[{'price': price_id, 'quantity': 1}],
            mode='subscription',
            success_url='http://localhost/checkout/success',
            cancel_url='http://localhost/checkout/cancel',
        )

    GET /checkout/cancel
        → 200 {"message": "Checkout cancelled. No charges were made."}

    If stripe_client is None, use _MockStripe().
    """
    # TODO: create app, implement /checkout POST and /checkout/cancel GET
    raise NotImplementedError


In [ ]:
def build_checkout_api(stripe_client=None) -> FastAPI:
    client = stripe_client if stripe_client is not None else _MockStripe()
    app    = FastAPI()

    class _CheckoutReq(BaseModel):
        plan: str

    @app.post("/checkout")
    def create_checkout(req: _CheckoutReq):
        if req.plan not in PLAN_PRICES:
            raise HTTPException(status_code=400,
                                detail=f"Unknown plan: {req.plan!r}")
        session = client.checkout.Session.create(
            payment_method_types=["card"],
            line_items=[{"price": PLAN_PRICES[req.plan], "quantity": 1}],
            mode="subscription",
            success_url="http://localhost/checkout/success",
            cancel_url="http://localhost/checkout/cancel",
        )
        return {"session_id": session["id"], "checkout_url": session["url"]}

    @app.get("/checkout/cancel")
    def cancel():
        return {"message": "Checkout cancelled. No charges were made."}

    return app


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # default (mock) client
    app = build_checkout_api()
    c   = TestClient(app, raise_server_exceptions=False)

    # valid plan → 200 with session_id and checkout_url
    r = c.post("/checkout", json={"plan": "pro"})
    assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
    d = r.json()
    assert "session_id"   in d, f"Missing session_id: {d}"
    assert "checkout_url" in d, f"Missing checkout_url: {d}"
    score += 1; print("\u2705 POST /checkout returns session_id and checkout_url")

    # session_id and url are non-empty strings
    assert isinstance(d["session_id"], str) and d["session_id"]
    assert isinstance(d["checkout_url"], str) and d["checkout_url"]
    score += 1; print("\u2705 session_id and checkout_url are non-empty strings")

    # unknown plan → 400
    r2 = c.post("/checkout", json={"plan": "enterprise"})
    assert r2.status_code == 400, f"Expected 400, got {r2.status_code}"
    score += 1; print("\u2705 unknown plan \u2192 400")

    # cancel endpoint
    r3 = c.get("/checkout/cancel")
    assert r3.status_code == 200
    assert "cancelled" in r3.json().get("message", "").lower()
    score += 1; print("\u2705 GET /checkout/cancel returns cancellation message")

    # custom stripe_client is used (not the default mock)
    class _CustomMock:
        class checkout:
            class Session:
                @staticmethod
                def create(**kwargs):
                    return {"id": "cs_custom_xyz", "url": "https://custom.stripe.com"}
    app2 = build_checkout_api(stripe_client=_CustomMock())
    c2   = TestClient(app2, raise_server_exceptions=False)
    r4   = c2.post("/checkout", json={"plan": "pro"})
    assert r4.json()["session_id"] == "cs_custom_xyz"
    score += 1; print("\u2705 custom stripe_client is injected and used")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_checkout_api(stripe_client=None) -> FastAPI:
    client = stripe_client if stripe_client is not None else _MockStripe()
    app    = FastAPI()

    class _CheckoutReq(BaseModel):
        plan: str

    @app.post("/checkout")
    def create_checkout(req: _CheckoutReq):
        if req.plan not in PLAN_PRICES:
            raise HTTPException(status_code=400,
                                detail=f"Unknown plan: {req.plan!r}")
        session = client.checkout.Session.create(
            payment_method_types=["card"],
            line_items=[{"price": PLAN_PRICES[req.plan], "quantity": 1}],
            mode="subscription",
            success_url="http://localhost/checkout/success",
            cancel_url="http://localhost/checkout/cancel",
        )
        return {"session_id": session["id"], "checkout_url": session["url"]}

    @app.get("/checkout/cancel")
    def cancel():
        return {"message": "Checkout cancelled. No charges were made."}

    return app
```

**Why inject the Stripe client?** The real `stripe` module makes network calls to Stripe's API. Tests should never make real API calls — they're slow, require credentials, and create test data. Duck-typing the client lets you swap the real Stripe SDK for a mock with zero code changes.

</details>